# 4. AIND Ephys Postprocessing

Run [aind-ephys-postprocessing](https://github.com/AllenNeuralDynamics/aind-ephys-postprocessing/tree/main/code) on the preprocessed recording (8 s window from `03_aind_ephys_spikesort_kilosort4.ipynb`) and the KS4 sorting.

The capsule expects `../data/` to contain both:
- `preprocessed_<name>/` directory + `binary_<name>.json`
- `spikesorted_<name>/` directory

It writes `postprocessed_<name>.zarr` (a SortingAnalyzer with extensions: random_spikes, waveforms, templates, spike_amplitudes, unit_locations, etc.) and a processing-metadata JSON into `../results/`.

Some default quality-metric parameters assume long recordings (e.g. `presence_ratio.bin_duration_s=60`); we override those for the 8-second toy.

## Imports and deps

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "scikit-learn", "numba",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 21 packages in 10ms
Uninstalled 2 packages in 5ms
Installed 2 packages in 2ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'scikit-learn', 'numba'], returncode=0)

## Clone the capsule and seed `data/`

We pull the preprocessed recording from the preprocessing capsule's `results/` (which was last written with the 8-second window inside the KS4 notebook) and the sorting from this folder's `output/03_spikesort_results/`.

In [3]:
post_repo = Path("/tmp/aind-ephys-postprocessing")
if not post_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-postprocessing.git",
            str(post_repo),
        ],
        check=True,
    )

data_dir = post_repo / "data"
results_dir = post_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    shutil.rmtree(stale) if stale.is_dir() else stale.unlink()

preproc_results = Path("/tmp/aind-ephys-preprocessing") / "results"
sort_results = Path.cwd() / "output/03_spikesort_results"
assert preproc_results.exists(), "Run 03_aind_ephys_spikesort_kilosort4.ipynb first."
assert sort_results.exists(), "Run 03_aind_ephys_spikesort_kilosort4.ipynb first."

for entry in preproc_results.iterdir():
    dest = data_dir / entry.name
    shutil.copytree(entry, dest) if entry.is_dir() else shutil.copy2(entry, dest)

for entry in sort_results.iterdir():
    if entry.name.startswith("spikesorted_"):
        dest = data_dir / entry.name
        shutil.copytree(entry, dest) if entry.is_dir() else shutil.copy2(entry, dest)

print("Seeded data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

Cloning into '/tmp/aind-ephys-postprocessing'...


Seeded data dir:
  binary_block0_None_recording1.json
  data_process_preprocessing_block0_None_recording1.json
  preprocessed_block0_None_recording1
  preprocessed_block0_None_recording1.json
  preprocessedviz_block0_None_recording1.json
  spikesorted_block0_None_recording1


## Write a toy params.json

Lower bin sizes so the metrics that assume minutes-long recordings still produce values for the 8-second toy.

In [4]:
upstream_params = json.loads((post_repo / "code" / "params.json").read_text())
params = {
    **upstream_params,
    "job_kwargs": {**upstream_params.get("job_kwargs", {}), "n_jobs": 1},
}
# Newer spikeinterface dropped `sparsity` from ComputeTemplateMetrics; strip it.
params["template_metrics"] = {
    k: v for k, v in upstream_params.get("template_metrics", {}).items() if k != "sparsity"
}
# `l_ratio` and `isolation_distance` were merged into `mahalanobis`; drop them.
renamed = {"l_ratio", "isolation_distance"}
params["quality_metrics_names"] = [
    name for name in upstream_params.get("quality_metrics_names", []) if name not in renamed
]
if "mahalanobis" not in params["quality_metrics_names"]:
    params["quality_metrics_names"].append("mahalanobis")
# Only keep per-metric params for metrics that are still in quality_metrics_names.
qm_keep = set(params["quality_metrics_names"])
filtered_qm = {
    k: v for k, v in upstream_params.get("quality_metrics", {}).items() if k in qm_keep
}
params["quality_metrics"] = {
    **filtered_qm,
    "presence_ratio": {"bin_duration_s": 1.0},
    "firing_range": {"bin_size_s": 1.0, "percentiles": [5, 95]},
    "amplitude_cv": {
        **filtered_qm.get("amplitude_cv", {}),
        "min_num_bins": 1,
        "average_num_spikes_per_bin": 5,
    },
}
params_path = post_repo / "code" / "params_toy.json"
params_path.write_text(json.dumps(params, indent=2))
print("Wrote", params_path)

# The capsule was written for spikeinterface < 0.104; newer versions renamed
# the `qm_params=` kwarg to `metric_params=`. Patch run_capsule.py in place.
capsule_py = post_repo / "code" / "run_capsule.py"
src = capsule_py.read_text()
src = src.replace("qm_params=quality_metrics_params", "metric_params=quality_metrics_params")
capsule_py.write_text(src)
print("Patched qm_params -> metric_params")

Wrote /tmp/aind-ephys-postprocessing/code/params_toy.json
Patched qm_params -> metric_params


## Run the postprocessing capsule

In [5]:
argv = [
    "python", "-u", "run_capsule.py",
    "--params", params_path.name,
    "--n-jobs", "1",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=post_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"postprocessing run failed with code {result.returncode}")

Running: python -u run_capsule.py --params params_toy.json --n-jobs 1


Running postprocessing with the following parameters:
	USE_MOTION_CORRECTED: False
	N_JOBS: -1

POSTPROCESSING
Found 0 json configurations
	Processing block0_None_recording1
	Loaded binary recording from JSON
	Loading lazy recording from JSON
	Creating sorting analyzer
NumExpr defaulting to 14 threads.
	Number of original units: 14 -- Number of units after de-duplication: 10
	Setting temporary binary recording
	Computing all postprocessing extensions
	Computing quality metrics
	Saving SortingAnalyzer to zarr
POSTPROCESSING time: 24.46s



## Copy outputs next to the notebook

In [6]:
local_results_dir = Path.cwd() / "output/04_postprocessing_results"
local_results_dir.parent.mkdir(parents=True, exist_ok=True)
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.iterdir()):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  data_process_postprocessing_block0_None_recording1.json  (2229 bytes)
  postprocessed_block0_None_recording1.zarr  (dir)


## Load the SortingAnalyzer and inspect quality metrics

In [7]:
import spikeinterface.full as si

zarr_dirs = [p for p in local_results_dir.iterdir() if p.name.startswith("postprocessed_") and p.suffix == ".zarr"]
print("Postprocessed analyzers:", [p.name for p in zarr_dirs])

if zarr_dirs:
    analyzer = si.load_sorting_analyzer(zarr_dirs[0])
    print(analyzer)
    print("Extensions computed:", sorted(analyzer.get_loaded_extension_names()))

    qm_ext = analyzer.get_extension("quality_metrics")
    if qm_ext is not None:
        qm = qm_ext.get_data()
        print()
        print(qm)

Postprocessed analyzers: ['postprocessed_block0_None_recording1.zarr']


SortingAnalyzer: 70 channels - 10 units - 1 segments - zarr - sparse
Loaded 13 extensions: correlograms, isi_histograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, spike_locations, template_metrics, template_similarity, templates, unit_locations, waveforms
Extensions computed: ['correlograms', 'isi_histograms', 'noise_levels', 'principal_components', 'quality_metrics', 'random_spikes', 'spike_amplitudes', 'spike_locations', 'template_metrics', 'template_similarity', 'templates', 'unit_locations', 'waveforms']

    amplitude_cutoff  amplitude_cv_median  amplitude_cv_range  \
0                NaN             0.103542            0.142882   
2                NaN             0.040664            0.077025   
3                NaN             0.575957            0.000000   
5                NaN             0.628303            0.359084   
8                NaN             0.267783            0.303546   
9                NaN             0.165905          